# 3 · Explore and plan

⏱ approx. 25 min before lunch + 15 min after lunch

**Stage of the pipeline:**
> **EXPLORE** → VALIDATE → REPORT & CLEAN → PACKAGE → DMP & DEPOSIT

We now switch from *learning the tools* to *using them*. For the rest of the day we follow one realistic workflow: a research project hands you a raw data file, and your job is to turn it into something **validated, documented, and ready to hand over or deposit**.

| Notebook | Stage | Tool |
|---|---|---|
| `3_explore-and-plan` | Explore & plan | assistant + `pandas` |
| `4_data-validation` | Validate | `pandera` |
| `5_quality-report` | Report & clean | `fg-data-profiling` |
| `6_metadata-packaging` | Describe & package | `frictionless` |
| `7_dmp-and-deposit` | DMP & deposit | assistant + Zenodo API |

Each tool is one item on the toolbelt. You don't need to memorise them - you need to know **what each one is for** and how to ask the assistant to use it correctly.

> Throughout the day the rule is **AI-first, but always understand and verify**:
> we let the assistant draft the code, then we *read it, run it, and check the result*.

In this notebook the assistant helps us to
* get a first impression of the raw data,
* draft a plan for the work ahead,
* generate exploration code, and
* generate a complete, documented exploration notebook.

## Initialize the AI assistant

This is the reusable initialization from notebook 2. It reads your endpoint URL, API key, model name and the **Data Steward system prompt** from your `secrets` file.

In [ ]:
# Import os to read env vars
import os
# Import bob and helper functions
from bia_bob import bob, available_models, fix, doc

# Initialize assistant, reads API key from env var automatically
bob.initialize(
    # read endpoint URL from env var
    endpoint=os.getenv('ENDPOINT_URL'),
    # read the model name from env var (set in your secrets file)
    model=os.getenv('LLM_MODEL'),
    # read the Data Steward system prompt from env var
    system_prompt=os.getenv('SYSTEM_PROMPT_DATA_STEWARD'),
)

Let's confirm the assistant is available and in its Data Steward role:

In [2]:
%bob Briefly, in two sentences: what is your role, and what can you help me with ?

I’m an AI assistant that supports research data stewards by providing policy‑aware, practical guidance across the full data lifecycle—from planning and documentation to FAIR‑compliant sharing, preservation, and reuse. I can help you draft and review data management plans, design metadata and naming standards, conduct data profiling and quality checks, choose appropriate repositories, and create actionable checklists, templates, and documentation tailored to your institutional, funder, and domain requirements.

## Our working data: a raw incoming dataset

Today we work with `../data/field_samples_raw.csv` - a small dataset of biological field samples that a project has handed over. It is *raw*: nobody has validated or cleaned it yet.

A **data dictionary** describing what the *clean* data should look like is provided in `../data/data_dictionary.md`. Open it now in JupyterLab (right-click > Open With > Markdown Preview) and keep it handy, it is our reference for "what good looks like".

Let's take a first look with [`pandas`](https://pandas.pydata.org/docs/). If `pandas` is not already available in your venv, you can use `uv` to install (add) it:

In [3]:
#!uv add pandas

If you do not understand what the following code does, you may ask your assistant ;)

In [4]:
import pandas as pd

df = pd.read_csv("../data/field_samples_raw.csv")

print("Shape (rows, columns):", df.shape)
df.head(10)

Shape (rows, columns): (182, 9)


,sample_id,species,island,date_collected,body_mass_g,flipper_length_mm,sex,region,notes
0,S0001,Adelie,Torgersen,2021-12-04,4547,201.2,.,Antarctica,juvenile?
1,S0002,Gentoo,NaN,2023-05-26,4650,212.4,Male,Antarctica,NaN
2,S0003,adelie,NaN,2022-10-09,2104,191.5,NaN,Antarctica,NaN
3,S0004,Chinstrap,Bisco,22/01/2021,5766,197.4,F,Antarctica,NaN
4,S0005,Chinstrap,Dream,2021-11-09,4814,219.9,F,Antarctica,NaN
5,S0006,Gentoo,Dream,2021-04-27,3959,203.0,.,Antarctica,collected near Pointe Géologie
6,S0007,Gentooo,NaN,5.8.2023,4168 g,182.6,F,Antarctica,NaN
7,S0008,Adelie,Dream,2021-01-28,4847 g,217.9,F,Antarctica,NaN
8,S0009,Gentoo,Torgersen,2021-11-24,3586,197.5,male,Antarctica,NaN
9,S0010,Adelie,Torgersen,4.9.2021,3667,199.6,M,Antarctica,NaN


**Take a moment to look.** Even from the first rows you can probably already spot trouble, e.g., in the `species`, `sex`, `body_mass_g`, or `date_collected` columns.

Let's ask the assistant for a quick first impression, using prompt augmentation via `{}`. However, always keep in mind the potential size of the augmented variable related to the available context size of the model behinf your assistant!

Notice we ask for a *short text answer, no code* - we just want orientation:

In [5]:
%%bob
Here is a raw dataset I just received: {df}
Give me a short, plain-language list of the most obvious data quality problems you notice.
Just a short bulleted text answer. No code, no cleaning yet.

- **Inconsistent species spelling/casing** (e.g., “Adelie”, “adelie”, “Gentoo”, “Gentooo”).
- **Missing values** in key columns such as `island`, `sex`, `body_mass_g`, and `notes`.
- **Mixed date formats** (`YYYY‑MM‑DD`, `DD/MM/YYYY`, `DD.MM.YYYY`) and ambiguous or malformed dates (e.g., “22/01/2021”, “28.3.2021”).
- **Placeholder or out‑of‑range values** (e.g., body mass of **‑999** as a sentinel for missing data).
- **Inconsistent sex codes** (e.g., “M”, “MALE”, “Male”, “F”, “.”, and missing values) and a dot placeholder.
- **Island names with typos or missing entries** (e.g., “NaN”, “Bisco” vs. “Biscoe”, “Dream”, “Torgersen”).
- **Notes column containing free‑text metadata** that may need standardisation (e.g., “weather: storm”, “juvenile?”).

Keep the assistant's list in mind, but remember to **verify** it against what you see. In the next notebooks we will systematically catch and fix these issues with the toolbelt, instead of eyeballing them.

## Drafting Plans

In [7]:
%%bob
I have received a raw data file in csv format from a research project. I want to explore and validate the data with the final goal to 
include the gained information in a DMP I need to write for an according data science project.
What are the steps and tasks I need to consider ?
Help me planning the data analysis.
Just a short text explaining the coarse plan for now. No code !
Store this text in a Markdown file "my-plan.md".

The file has been saved as [my-plan.md](my-plan.md).

Open `my-plan.md` (right-click > Open With > Markdown Preview) and read it critically: is anything missing, is anything irrelevant for *this* dataset?
An example of such a plan is in `../examples/my-plan.md`.

---
## Generating Code

We can now use our AI assistant to tackle the separate steps, e.g., basic data exploration and profiling. Let's ask for according Python code:

In [ ]:
%%bob
Generate Python code in a code cell below to do the following:
* Read the file in "../data/field_samples_raw.csv"
* Explore basic data properties and statistics, include missing values and outliers
* Keep the code as short as possible
* Use short and precise commments
* Do not assume any values or column names but read and use the ones provided with the data

Run the generated code. Below is a **hand-written reference** for comparison. Which of the problems from the assistant's first impression does the code reveal - and which ones does it miss?

In [8]:
# Reference exploration of ../data/field_samples_raw.csv (written by hand, for comparison)
import pandas as pd

df = pd.read_csv("../data/field_samples_raw.csv")

print("Shape (rows, cols):", df.shape)
print("\nColumn types as inferred by pandas:")
print(df.dtypes.to_string())

print("\nMissing values per column (only what pandas recognises as missing):")
print(df.isna().sum().to_string())
print("\nExact duplicate rows:", df.duplicated().sum())

# Text columns with few distinct values -> candidates for controlled vocabularies
print("\nDistinct values in categorical-looking columns:")
for col in df.columns:
    if df[col].dtype == object and df[col].nunique() <= 15:
        print(f"  {col}: {sorted(df[col].dropna().unique().tolist())}")

# Columns that are *mostly* numbers, but were not read as numbers
as_num = df.apply(pd.to_numeric, errors="coerce")
mostly_numeric = [c for c in df.columns if as_num[c].notna().mean() > 0.8]
print("\nColumns that are mostly numeric:", mostly_numeric)
print(as_num[mostly_numeric].describe().T.round(1))

Shape (rows, cols): (182, 9)

Column types as inferred by pandas:
sample_id            object
species              object
island               object
date_collected       object
body_mass_g          object
flipper_length_mm    object
sex                  object
region               object
notes                object

Missing values per column (only what pandas recognises as missing):
sample_id              0
species                0
island                43
date_collected         9
body_mass_g            8
flipper_length_mm      8
sex                   21
region                 0
notes                161

Exact duplicate rows: 2

Distinct values in categorical-looking columns:
  species: [' Adelie', 'ADELIE', 'Adelie', 'Adelie ', 'Chinstrap', 'Chinstrap Penguin', 'Gentoo', 'Gentooo', 'adelie', 'chinstrap', 'gentoo']
  island: ['Bisco', 'Biscoe', 'Dream', 'Torgersen']
  sex: ['.', 'F', 'FEMALE', 'Female', 'M', 'MALE', 'Male', 'female', 'male', 'unknown']
  region: ['Antarctica']
  notes

Note what `pandas` did **not** tell us: `body_mass_g` and `flipper_length_mm` were read as text, and `-999`, `NULL` or `?` count as ordinary values, not as missing.
We come back to exactly this in notebook 4.

---
## Generating Notebooks

Now we want a more detailed data exploration within a dedicated Jupyter notebook, including sufficient documentation and explanation.

> ⏱ **Start the next two cells right before the lunch break.** Generating a whole notebook can take several minutes, especially when many people use the endpoint at the same time.

For this, we may also re-initialize our AI assistant and use a model more specialized for coding.  
Note: more sophisticated and bigger models may need longer to generate the answer.

In [ ]:
# Initialize assistant, reads API key from env var automatically
bob.initialize(
    # read endpoint URL from env var
    endpoint=os.getenv('ENDPOINT_URL'),
    # select an available model for your purpose
    model=os.getenv('LLM_MODEL_CODER'),
    # read system prompt from env var
    system_prompt=os.getenv('SYSTEM_PROMPT_DATA_STEWARD'),
)

In [ ]:
%%bob
Generate a Jupyter notebook called "field_samples_exploration", which does the following:
* Read the file in "../data/field_samples_raw.csv"
* Explore the data properties and statistics, include missing values and outliers
* Visualize basic data distributions
* Do not assume any values or column names but read and use the ones provided with the data
* Structure the notebook so that it is clear and self-explanatory
* Provide Markdown cells for documentation which explain each step and why it is necessary
* Provide a summary of the exploration and its findings at the end of the notebook - the summary shall be included in a DMP

---
## After lunch: review the generated notebook

Open `field_samples_exploration.ipynb` and run it with `Kernel > Restart Kernel and Run All Cells...`. Then check:

* Does it run from top to bottom without errors? If not, try `%%fix` on the failing cell.
* Does it hard-code column names or values, although we asked it not to?
* How does it handle `"4168 g"`, `"195,5"` and `-999`? Silently, or does it report them?
* Does the summary at the end only contain claims that the notebook actually shows?

A hand-written reference is in `../examples/field_samples_exploration.ipynb`.

> A generated notebook is a **draft**. It becomes documentation only after you have read it, run it, and corrected it.

## What's next

We continue with **`4_data-validation.ipynb`**, where we turn the data dictionary into an enforceable schema and let it tell us exactly which records break the rules.